In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import RMSprop
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import OneClassSVM
import matplotlib.pyplot as plt
import seaborn as sns


# --- Global MIT Aesthetic Settings ---
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.labelweight'] = 'bold'

# -<Do not be annoying>
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='statsmodels')
warnings.filterwarnings("ignore", category=FutureWarning, module='statsmodels')


2026-04-19 09:21:23.899902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776579684.052811   30007 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776579684.106475   30007 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-19 09:21:24.262499: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import RMSprop
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import OneClassSVM
import matplotlib.pyplot as plt
import seaborn as sns

# Fix for NixOS/Micromamba environment
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/home/molderon/micromamba/envs/Kaon'

class BreathAnalysisPipeline:
    def __init__(self, input_shape, learning_rate=0.001):
        self.input_shape = input_shape
        self.model = None
        self.svm = OneClassSVM(kernel='rbf', nu=0.0000001, gamma=0.15)
        self.learning_rate = learning_rate
        self.num_classes = 0

    def _build_cnn(self, num_classes):
        model = models.Sequential([
            layers.Conv1D(64, 5, activation='relu', input_shape=self.input_shape),
            layers.MaxPooling1D(3),
            layers.Conv1D(128, 5, activation='relu'),
            layers.GlobalMaxPooling1D(),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(num_classes, activation='softmax')
        ])
        model.compile(
            optimizer=RMSprop(learning_rate=self.learning_rate), 
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    def load_and_format(self, csv_path):
        print(f"Loading and Cleaning data from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        clean_df = df[df['Cluster_Label'] != -1].copy()
        
        unique_labels = sorted(clean_df['Cluster_Label'].unique())
        label_map = {old: new for new, old in enumerate(unique_labels)}
        clean_df['Mapped_Label'] = clean_df['Cluster_Label'].map(label_map)
        
        self.num_classes = len(unique_labels)
        print(f"Detected {self.num_classes} valid clusters (excluding noise).")

        X, y, raw_features = [], [], []
        
        for breath_id, group in clean_df.groupby("Breath_ID"):
            feat = group[['Flow', 'Pressure']].values
            
            # Interpolate to match CNN input_shape
            if len(feat) != self.input_shape[0]:
                indices = np.linspace(0, len(feat)-1, self.input_shape[0]).astype(int)
                feat = feat[indices]
            
            X.append(feat)
            y.append(group['Mapped_Label'].iloc[0])
            raw_features.append(feat.flatten())
            
        return np.array(X), np.array(y), np.array(raw_features)

    def train_cnn(self, X, y, epochs=99):
        # Build model now that we know the true number of classes
        self.model = self._build_cnn(self.num_classes)
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        print(f"Training CNN on {self.num_classes} classes...")
        self.history = self.model.fit(
            X_train, y_train, 
            epochs=epochs, 
            validation_data=(X_test, y_test), 
            batch_size=32
        )
        return X_test, y_test

    def fit_svm(self, data):
        print("Fitting One-Class SVM on all data (including noise)...")
        self.svm.fit(data)
        return self.svm.predict(data)

    def plot_hysteresis_cycle(self, data, counters=None):
        plt.figure(figsize=(8, 5))
        plt.plot(data[:, 0], label='Flow', color='teal')
        plt.plot(data[:, 1], label='Pressure', color='orange')
        if counters:
            for c in counters: plt.axvline(c, c='r', ls='--')
        plt.legend()
        plt.title("Hysteresis Cycle Representation")
        plt.show()

# --- Execution ---

# 1. Initialize (Sequence length 200, 2 features)
pipeline = BreathAnalysisPipeline(input_shape=(200, 2))

# 2. Load and Auto-fix labels (Filters -1 and remaps 0 -> N-1)
X, y, raw_svm_data = pipeline.load_and_format("Dataset_Labeled.csv")

# 3. Train
X_test, y_test = pipeline.train_cnn(X, y, epochs=30)

# 4. SVM Outlier Detection
svm_preds = pipeline.fit_svm(raw_svm_data)

# 5. Visualization
pipeline.plot_hysteresis_cycle(X[0], counters=[131, 296])

Loading and Cleaning data from Dataset_Labeled.csv...
Detected 70 valid clusters (excluding noise).


> # Bi-Driectopmal 1d ConvNet
>

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
import matplotlib.pyplot as plt

# Fix for NixOS/Micromamba environment
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/home/molderon/micromamba/envs/Kaon'

class HysteresisAnalysis:
    def __init__(self, input_shape, learning_rate=0.001):
        self.input_shape = input_shape # Expected (200, 2)
        self.learning_rate = learning_rate
        self.model = None
        self.svm = OneClassSVM(kernel='rbf', nu=0.0000001, gamma=0.15)

    def _build_hybrid_model(self, X_sample):
        """
        Implements the requested CNN + BiLSTM architecture.
        Note: Changed Dense(1) to Dense(num_classes) if you still want classification,
        but kept as Dense(1) per your request for Regression.
        """
        # 1. Normalization Layer
        norm_layer = layers.Normalization(input_shape=self.input_shape, axis=-1)
        norm_layer.adapt(X_sample)

        model = models.Sequential([
            norm_layer,
            layers.Conv1D(128, 3, activation="relu"),
            layers.MaxPooling1D(),
            layers.BatchNormalization(),
            
            layers.Conv1D(256, 3, activation="relu"),
            layers.MaxPooling1D(),
            layers.BatchNormalization(),
            
            # Bidirectional LSTMs for temporal context
            layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
            layers.Bidirectional(layers.LSTM(128, return_sequences=True)),
            
            layers.GlobalAveragePooling1D(),
            layers.Dropout(0.5),
            layers.Dense(1) # Regression Exit
        ])

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss="mae"
        )
        return model

    def load_and_format(self, csv_path):
        print(f"Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        # We include all data (including -1) for regression/SVM
        X, y, raw_features = [], [], []
        
        for breath_id, group in df.groupby("Breath_ID"):
            feat = group[['Flow', 'Pressure']].values
            
            # Interpolate to target length
            if len(feat) != self.input_shape[0]:
                indices = np.linspace(0, len(feat)-1, self.input_shape[0]).astype(int)
                feat = feat[indices]
            
            X.append(feat)
            y.append(group['Cluster_Label'].iloc[0]) # Target for regression
            raw_features.append(feat.flatten())
            
        return np.array(X), np.array(y), np.array(raw_features)

    def train(self, X, y, epochs=30):
        # Build and adapt model to the specific training data
        self.model = self._build_hybrid_model(X)
        self.model.summary()
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        print("Training Hybrid CNN-BiLSTM...")
        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=epochs,
            batch_size=32
        )
        return X_test, y_test

    def fit_svm(self, data):
        print("Fitting One-Class SVM...")
        self.svm.fit(data)
        return self.svm.predict(data)

    def plot_results(self, X_sample):
        # Specific plotting logic for Flow/Pressure as requested
        plt.figure(figsize=(10, 4))
        plt.plot(X_sample[:, 0], label='Flow', color='blue', alpha=0.7)
        plt.plot(X_sample[:, 1], label='Pressure', color='orange', alpha=0.7)
        plt.title("Hysteresis Input Visualization")
        plt.legend()
        plt.show()

# --- Execution ---

# Initialize for (200 time steps, 2 features)
pipeline = HysteresisAnalysis(input_shape=(200, 2))

# Load data
X, y, raw_svm_data = pipeline.load_and_format("Dataset_Labeled.csv")

# Train (includes Norm.adapt inside)
X_test, y_test = pipeline.train(X, y, epochs=30)

# SVM Analysis
svm_preds = pipeline.fit_svm(raw_svm_data)

# Visualization of the first breath
pipeline.plot_results(X[0])